# 02 — Full experiments (Kaggle GPU)

Runs the workloads that do not fit the 4 GB laptop: the PatchCore reference
configuration, the ablation sweep, autoencoder training, and the robustness grid.

**Kaggle settings**

| setting | value |
|---|---|
| Accelerator | GPU P100 (16 GB) — or T4 x2 |
| Internet | **On** (needed to fetch VisA and the package) |
| Session limit | 12 h interactive, ~9 h on commit |
| Weekly quota | ~30 GPU-hours |
| Working dir | 20 GB, `/kaggle/working` |

**Discipline (docs/04 §2).** This notebook is a thin driver: it installs the
package and calls its API. No model code lives in a cell. Every stage
checkpoints to `/kaggle/working`, because the session can die at any moment and
a sweep that cannot resume is a sweep that gets run twice.

Dataset: **VisA** (Amazon), CC BY 4.0 — see [ATTRIBUTION.md](../ATTRIBUTION.md).

## 0. Environment and package

In [ ]:
import os, sys, subprocess, pathlib, json, time

ON_KAGGLE = pathlib.Path('/kaggle').exists()
WORK = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd() / 'kaggle_local'
WORK.mkdir(parents=True, exist_ok=True)
print('kaggle    :', ON_KAGGLE)
print('working   :', WORK)

import torch
print('torch     :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu       :', p.name, f'{p.total_memory / 1024**3:.1f} GB')

### Getting the package onto Kaggle

Three routes, in order of preference. Set `REPO_URL` if the repository is
pushed; otherwise attach it as a Kaggle dataset (Add Data -> Upload).

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<user>/mvtec-visual-inspector'
PKG_DATASET = '/kaggle/input/mvtec-visual-inspector'  # if uploaded as a dataset

def install_package():
    if REPO_URL:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        f'git+{REPO_URL}'], check=True)
        return 'pip install from git'
    if pathlib.Path(PKG_DATASET).exists():
        sys.path.insert(0, str(pathlib.Path(PKG_DATASET) / 'src'))
        return f'sys.path from {PKG_DATASET}'
    local = pathlib.Path.cwd()
    for candidate in (local, local.parent):
        if (candidate / 'src' / 'inspector').exists():
            sys.path.insert(0, str(candidate / 'src'))
            return f'sys.path from {candidate}'
    raise RuntimeError(
        'inspector package not found. Set REPO_URL, or attach the repo as a '
        'Kaggle dataset at ' + PKG_DATASET)

print(install_package())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'opencv-python-headless', 'scikit-image'], check=False)

from inspector.utils.env import capture
env = capture()
print('git       :', env['git_sha'], '(dirty)' if env['dirty'] else '(clean)')

## 1. Data

Either attach VisA as a Kaggle dataset, or download it here (1.8 GB, needs
Internet on). The download path re-verifies its SHA-256 manifest, which is what
makes a silently truncated transfer detectable.

In [ ]:
from inspector.fetch import fetch

VISA_INPUT = pathlib.Path('/kaggle/input/visa-anomaly/VisA_20220922')
if VISA_INPUT.exists():
    VISA_ROOT = VISA_INPUT
    print('using attached dataset:', VISA_ROOT)
else:
    def progress(done, total):
        pct = int(100 * done / total) if total else 0
        if pct % 10 == 0 and done % (200 << 20) < (1 << 20):
            print(f'  {pct:3d}%  {done / 1e9:.2f} / {total / 1e9:.2f} GB')
    manifest = fetch('visa', WORK / 'data', extract=True, progress=progress)
    VISA_ROOT = pathlib.Path(manifest.extracted_to)
    print('downloaded to:', VISA_ROOT)

assert (VISA_ROOT / 'split_csv' / '1cls.csv').exists(), 'official split CSV missing'

In [ ]:
from inspector.data import ensure_validation, load_category
from inspector.data.integrity import audit

CATEGORIES = ['pcb1', 'macaroni2', 'capsules']
indices = {}
for cat in CATEGORIES:
    idx = load_category(VISA_ROOT, cat, layout='visa')
    idx, _ = ensure_validation(idx, val_fraction=0.15, seed=0)
    indices[cat] = idx
    ok = all(r.ok for r in audit(idx))
    print(f'{cat:<11} ' + '  '.join(f'{k}={len(v)}' for k, v in sorted(idx.items())),
          '| integrity', 'OK' if ok else 'FAILED')
    assert ok, f'{cat}: split integrity failed'

## 2. Resumable experiment runner

Every completed run is appended to a JSONL checkpoint keyed by its
configuration. A restarted session skips what is already done. Without this a
12-hour session limit turns a sweep into a gamble.

In [ ]:
import tempfile
from inspector.data.transforms import ImageTransform
from inspector.models import build_model
from inspector.pipeline import run_experiment
from inspector.results import append_results, render_markdown
from inspector.utils.seed import seed_everything

CHECKPOINT = WORK / 'runs.jsonl'
RESULTS_CSV = WORK / 'results_visa.csv'

def done_keys():
    if not CHECKPOINT.exists():
        return set()
    with open(CHECKPOINT, encoding='utf-8') as fh:
        return {json.loads(line)['key'] for line in fh if line.strip()}

def execute(key, category, model_name, seed, resolution, **kwargs):
    """Run one configuration unless the checkpoint says it is already done."""
    if key in done_keys():
        print(f'  skip (done): {key}')
        return None
    seed_everything(seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    transform = ImageTransform(mode='aspect_preserving', long_side=resolution)
    model = build_model(model_name, transform, seed=seed, **kwargs)
    started = time.perf_counter()
    with tempfile.TemporaryDirectory() as tmp:
        result = run_experiment(model, indices[category], workdir=tmp,
                                seed=seed, test_split='test')
    result.notes = key
    append_results([result], RESULTS_CSV)
    with open(CHECKPOINT, 'a', encoding='utf-8') as fh:
        fh.write(json.dumps({'key': key, 'elapsed_s': round(time.perf_counter() - started, 1),
                             'image_auroc': result.image_auroc,
                             'aupro_005': result.aupro_005}) + '\n')
    print(f'  {key}\n    {result.summary()}')
    return result

## 3. PatchCore reference configuration

WideResNet50-2, `layer2 + layer3`, 3x3 aggregation, 1% greedy coreset, k=1.
This is the row every ablation is measured against, so it is frozen and run with
3 seeds — the coreset start point is random, so a single run reports noise as
signal.

In [ ]:
REFERENCE = dict(backbone='wide_resnet50_2', layers=('layer2', 'layer3'),
                 patch_pool=3, projection_dim=1024, coreset_ratio=0.01,
                 k=1, batch_size=8)
RESOLUTION = 320
SEEDS = [0, 1, 2]

reference_runs = []
for cat in CATEGORIES:
    for seed in SEEDS:
        key = f'ref|{cat}|{RESOLUTION}|s{seed}'
        r = execute(key, cat, 'patchcore', seed, RESOLUTION, **REFERENCE)
        if r:
            reference_runs.append(r)

In [ ]:
import statistics

print(f"{'category':<12}{'I-AUROC':>18}{'AU-PRO@0.05':>20}{'SegF1':>16}")
print('-' * 66)
for cat in CATEGORIES:
    rows = [r for r in reference_runs if r.category == cat]
    if len(rows) < 2:
        continue
    def ms(attr):
        vals = [getattr(r, attr) for r in rows]
        return f'{statistics.mean(vals):.4f} +/- {statistics.stdev(vals):.4f}'
    print(f'{cat:<12}{ms("image_auroc"):>18}{ms("aupro_005"):>20}{ms("segf1_3sigma"):>16}')
print()
print('Report mean +/- std, never a single seed (protocol section 4.2).')

## 4. Ablation sweep — staged, not a grid

A full grid over backbone x layers x resolution x coreset x k is ~10^4 runs and
is not affordable. A staged sweep varies one axis at a time from the reference,
which captures the main effects in ~60 runs; the one interaction worth having is
re-varied jointly afterwards.

Single seed during the sweep, then 3 seeds for the winners. Spending 3 seeds on
a configuration that loses by 10 points buys nothing.

In [ ]:
AXES = {
    'backbone':   [dict(backbone=b) for b in ['resnet18', 'resnet50', 'wide_resnet50_2']],
    'layers':     [dict(layers=l) for l in [('layer2',), ('layer3',),
                                            ('layer2', 'layer3'),
                                            ('layer2', 'layer3', 'layer4')]],
    'coreset':    [dict(coreset_ratio=c) for c in [0.001, 0.01, 0.1, 0.25]],
    'k':          [dict(k=k) for k in [1, 3, 9]],
    'projection': [dict(projection_dim=d) for d in [128, 384, 1024]],
}
RESOLUTIONS = [224, 320, 448]

planned = sum(len(v) for v in AXES.values()) * len(CATEGORIES) + len(RESOLUTIONS) * len(CATEGORIES)
print(f'planned runs: {planned}  (single seed)')
print('At roughly 1-3 min each on a P100, budget accordingly against the 30 h/week quota.')

In [ ]:
sweep = []
for axis, variants in AXES.items():
    print(f'=== axis: {axis} ===')
    for variant in variants:
        label = ','.join(f'{k}={v}' for k, v in variant.items())
        for cat in CATEGORIES:
            cfg = {**REFERENCE, **variant}
            r = execute(f'{axis}|{label}|{cat}|{RESOLUTION}|s0', cat,
                        'patchcore', 0, RESOLUTION, **cfg)
            if r:
                sweep.append((axis, label, r))

In [ ]:
print('=== axis: resolution ===')
for res in RESOLUTIONS:
    for cat in CATEGORIES:
        r = execute(f'resolution|{res}|{cat}|s0', cat, 'patchcore', 0, res, **REFERENCE)
        if r:
            sweep.append(('resolution', str(res), r))

In [ ]:
# Effect of each axis, averaged over categories, against the reference.
baseline = {r.category: r for r in reference_runs if r.seed == 0}

print(f"{'axis':<12}{'variant':<34}{'d AU-PRO@0.05':>15}{'d I-AUROC':>12}")
print('-' * 74)
from collections import defaultdict
grouped = defaultdict(list)
for axis, label, r in sweep:
    if r.category in baseline:
        grouped[(axis, label)].append((r.aupro_005 - baseline[r.category].aupro_005,
                                       r.image_auroc - baseline[r.category].image_auroc))
for (axis, label), deltas in grouped.items():
    d_pro = statistics.mean(d[0] for d in deltas)
    d_auc = statistics.mean(d[1] for d in deltas)
    print(f'{axis:<12}{label:<34}{d_pro:>+15.4f}{d_auc:>+12.4f}')

### Adoption decision (DR-1)

A change is adopted only if it improves AU-PRO@0.05 on at least 2 of 3
categories, does not regress the third by more than 1 point, passes a paired
test after Holm-Bonferroni correction, and stays inside the latency and VRAM
budgets. Fill the verdict column at the time of the run, not afterwards.

In [ ]:
from inspector.stats import compare_models, holm_bonferroni

# Paired over categories here; with per-image AUPIMO available this becomes a
# per-image paired test, which has far more power. Recorded as a limitation.
family = len(grouped)
print(f'family size for the correction: {family}')
print()
for (axis, label), deltas in sorted(grouped.items(), key=lambda kv: -statistics.mean(d[0] for d in kv[1])):
    improved = sum(1 for d in deltas if d[0] > 0)
    worst = min(d[0] for d in deltas)
    verdict = 'adopt' if (improved >= 2 and worst > -0.01) else 'reject'
    print(f'{axis:<12}{label:<34}improved {improved}/{len(deltas)}  worst {worst:+.4f}  -> {verdict}')

## 5. Tier 1 — autoencoder baseline

The scope's required from-scratch baseline. Its purpose is to quantify what the
pretrained prior in PatchCore is actually worth, in points, on this data.

The L2-vs-SSIM comparison is the instructive part: L2 blurs, and a blurry
reconstruction produces error everywhere rather than at the defect.

In [ ]:
AE_CONFIGS = [
    dict(loss='l2',      latent_dim=128, residual='multiscale'),
    dict(loss='ssim',    latent_dim=128, residual='multiscale'),
    dict(loss='l2+ssim', latent_dim=128, residual='multiscale'),
    dict(loss='ssim',    latent_dim=32,  residual='multiscale'),
    dict(loss='ssim',    latent_dim=128, residual='raw'),
]

ae_runs = []
for cfg in AE_CONFIGS:
    label = ','.join(f'{k}={v}' for k, v in cfg.items())
    for cat in CATEGORIES:
        for seed in SEEDS:
            r = execute(f'ae|{label}|{cat}|s{seed}', cat, 'cae', seed, 256,
                        epochs=80, batch_size=16, early_stopping_patience=15, **cfg)
            if r:
                ae_runs.append(r)

In [ ]:
print(f"{'config':<40}{'I-AUROC':>10}{'AU-PRO@0.05':>14}{'P-AUROC':>10}")
print('-' * 74)
by_config = defaultdict(list)
for r in ae_runs:
    by_config[r.notes.split('|')[1]].append(r)
for label, rows in by_config.items():
    print(f'{label:<40}{statistics.mean(r.image_auroc for r in rows):>10.4f}'
          f'{statistics.mean(r.aupro_005 for r in rows):>14.4f}'
          f'{statistics.mean(r.pixel_auroc for r in rows):>10.4f}')

### What the pretrained prior bought

The comparison the whole ladder exists to make. Report it in points, and report
it per category — where the two disagree, that disagreement *is* the finding.

In [ ]:
best_ae = {}
for cat in CATEGORIES:
    rows = [r for r in ae_runs if r.category == cat]
    if rows:
        best_ae[cat] = max(rows, key=lambda r: r.aupro_005)

print(f"{'category':<12}{'best AE':>12}{'PatchCore':>12}{'delta':>10}")
print('-' * 46)
for cat in CATEGORIES:
    if cat in best_ae and cat in baseline:
        a, p = best_ae[cat].aupro_005, baseline[cat].aupro_005
        print(f'{cat:<12}{a:>12.4f}{p:>12.4f}{p - a:>+10.4f}')

## 6. Robustness

The model is fitted once on clean data and **never re-thresholded**. Corrupting
the test set and then re-tuning the threshold would answer a different and much
easier question.

The number to watch is not the AUROC drop. It is the **realized false-alarm rate
at the frozen threshold**: a model can hold its AUROC while its alarm rate goes
from 1% to 30%, because the whole score distribution shifted. Ranking survives;
the product does not.

> The corruption suite ([docs/05](../docs/05-robustness-protocol.md)) is specified
> but not yet implemented — it is Phase P7. This section is the harness; fill it
> in when `inspector.robustness` lands.

In [ ]:
try:
    from inspector.robustness import CORRUPTIONS, apply_corruption
    HAVE_ROBUSTNESS = True
except ImportError:
    HAVE_ROBUSTNESS = False
    print('inspector.robustness not implemented yet (Phase P7).')
    print('Specification: docs/05-robustness-protocol.md')
    print()
    print('Planned grid: 6 corruption families x 5 severities x 3 categories x 3 models')
    print('             = 270 inference-only evaluations, reusing the fitted models.')

## 7. Persist everything before the session dies

Kaggle discards `/kaggle/working` when the session ends unless the notebook is
committed or the outputs are saved. Download `results_visa.csv` and `runs.jsonl`,
then merge them into the repository's `reports/`.

In [ ]:
import shutil

print('results :', RESULTS_CSV, RESULTS_CSV.exists())
print('runs    :', CHECKPOINT, CHECKPOINT.exists())
if CHECKPOINT.exists():
    with open(CHECKPOINT, encoding='utf-8') as fh:
        rows = [json.loads(line) for line in fh if line.strip()]
    total = sum(r['elapsed_s'] for r in rows)
    print(f'completed runs : {len(rows)}')
    print(f'gpu time       : {total / 3600:.2f} h  of the ~30 h weekly quota')

summary = WORK / 'session_summary.md'
with open(summary, 'w', encoding='utf-8') as fh:
    fh.write('# Kaggle session\n\n')
    fh.write(f'- categories: {CATEGORIES}\n')
    fh.write(f'- reference resolution: {RESOLUTION}\n')
    fh.write(f'- completed runs: {len(rows) if CHECKPOINT.exists() else 0}\n')
    fh.write(f'- gpu hours: {total / 3600:.2f}\n' if CHECKPOINT.exists() else '')
    fh.write('\nData: VisA (c) Amazon, CC BY 4.0. No images redistributed.\n')
print('\nwrote', summary)

In [ ]:
# The full table. Every caption obligation from docs/02 is satisfied by
# render_markdown: split, threshold source, and oracle marking.
all_runs = reference_runs + [r for _, _, r in sweep] + ae_runs
print(render_markdown(all_runs))